# Experiment 1: Tokenization Analysis and Text Generation Using Hugging Face Transformers

**Course:** HAAI302COM - Artificial Intelligence Laboratory

## 1. Aim

To explore different tokenization strategies, including Byte Pair Encoding (BPE) and WordPiece, using the `tiktoken` and Hugging Face `tokenizers` libraries, and to perform text generation using a pre-trained GPT-2 model by studying the effect of different decoding parameters.

## 2. Objectives

- To understand the concept of tokenization in Natural Language Processing (NLP).
- To compare BPE and WordPiece tokenization strategies.
- To tokenize text using the `tiktoken` and Hugging Face libraries.
- To load a pre-trained GPT-2 language model.
- To generate text using different decoding parameters.
- To analyze the effect of temperature, top-k, top-p, and maximum generated tokens on the output.

## 3. Software Requirements

- Python 3.9 or later
- Google Colab or Jupyter Notebook
- Internet connection for downloading pre-trained models
- Required Python libraries: `transformers`, `tokenizers`, `tiktoken`, and `torch`

## 4. Theory

### A. Tokenization

Tokenization is the process of converting text into smaller units called tokens. These tokens may be words, subwords, characters, or byte-level units. Language models process token IDs rather than raw text.

Example input: `Artificial Intelligence is useful.`
Possible tokenization: `["Artificial", " Intelligence", " is", " useful", "."]`. The exact tokens depend on the tokenizer used.

### B. Byte Pair Encoding (BPE)

BPE is a subword tokenization technique that builds a vocabulary by repeatedly merging frequent character or byte sequences. It can represent common words as single tokens and split less common words into smaller units.

- Used by GPT-2 and several other language models.
- Handles many unfamiliar words by splitting them into subword units.
- The `tiktoken` library provides fast BPE-based tokenization for supported model encodings.

### C. WordPiece

WordPiece is a subword tokenization technique used by models such as BERT. It builds subword vocabularies and represents words using the available vocabulary pieces. For example, an unfamiliar word may be split into smaller pieces such as `play` and `##ing`, depending on the tokenizer vocabulary.

### D. Text Generation Using GPT-2

GPT-2 is a pre-trained autoregressive language model. It predicts the next token based on the tokens that came before it. This process repeats until a stopping condition is reached. For the prompt `Artificial Intelligence will change the future because`, the model generates a continuation based on the prompt.

### E. Decoding Parameters

| Parameter | Description |
|---|---|
| `max_new_tokens` | Controls the maximum number of new tokens generated. |
| `temperature` | Controls randomness. Lower values generally produce more predictable text; higher values produce more varied text. |
| `top_k` | Restricts selection to the top k likely next tokens. |
| `top_p` | Selects from a group of likely tokens whose cumulative probability reaches the specified threshold. |
| `do_sample` | Enables sampling when set to True; otherwise, generation is greedy. |
| `num_return_sequences` | Specifies the number of generated outputs. |

## 5. Algorithm / Procedure

1. Install the required Python libraries.
2. Import the necessary libraries.
3. Provide a sample sentence for tokenization.
4. Apply BPE tokenization using `tiktoken`.
5. Apply WordPiece tokenization using the Hugging Face BERT tokenizer.
6. Display the tokens, token IDs, and number of tokens.
7. Load the pre-trained GPT-2 tokenizer and language model.
8. Provide a text prompt as input to the model.
9. Generate text using different decoding parameters.
10. Compare the generated outputs and record observations.

## 6. Python Program

### Step 1: Install the required libraries

In [1]:
!pip install -q transformers tokenizers tiktoken torch

### Step 2: Analyze BPE and WordPiece tokenization

In [2]:
import tiktoken
from transformers import AutoTokenizer

text = "Artificial Intelligence is transforming education."

# BPE tokenization using tiktoken
bpe_tokenizer = tiktoken.get_encoding("gpt2")
bpe_ids = bpe_tokenizer.encode(text)
bpe_tokens = [bpe_tokenizer.decode([i]) for i in bpe_ids]

print("Original Text:", text)
print("\nBPE Tokens:", bpe_tokens)
print("BPE Token IDs:", bpe_ids)
print("Number of BPE Tokens:", len(bpe_ids))

Original Text: Artificial Intelligence is transforming education.

BPE Tokens: ['Art', 'ificial', ' Intelligence', ' is', ' transforming', ' education', '.']
BPE Token IDs: [8001, 9542, 9345, 318, 25449, 3707, 13]
Number of BPE Tokens: 7


In [3]:
# WordPiece tokenization using Hugging Face
wordpiece_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
wp_encoded = wordpiece_tokenizer(text, add_special_tokens=False)
wp_tokens = wordpiece_tokenizer.convert_ids_to_tokens(wp_encoded["input_ids"])

print("WordPiece Tokens:", wp_tokens)
print("WordPiece Token IDs:", wp_encoded["input_ids"])
print("Number of WordPiece Tokens:", len(wp_tokens))

WordPiece Tokens: ['artificial', 'intelligence', 'is', 'transforming', 'education', '.']
WordPiece Token IDs: [7976, 4454, 2003, 17903, 2495, 1012]
Number of WordPiece Tokens: 6


**Expected observation:** The two tokenizers may split the same sentence differently because they use different vocabularies and tokenization rules.

### Step 3: Load a pre-trained GPT-2 model

In [4]:
import torch
from transformers import AutoModelForCausalLM, set_seed

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Model loaded successfully!")
print("Device:", device)

Model loaded successfully!
Device: cpu


### Step 4: Generate text using different decoding parameters

A small helper function runs `model.generate()` with the chosen settings, so each experiment is a single line. A fixed random seed makes the sampled outputs repeatable.

In [5]:
prompt = "Artificial Intelligence will change the future because"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

def generate(**params):
    set_seed(42)  # same seed for every experiment
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=40, pad_token_id=tokenizer.eos_token_id, **params)
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [6]:
# Experiment 1: Greedy decoding
out1 = generate(do_sample=False)
print("Greedy Decoding:")
print(out1)

Greedy Decoding:
Artificial Intelligence will change the future because it will be able to do things that are not possible before.

The future of AI is not a question of whether AI will be able to do things that are not possible before. It is


In [7]:
# Experiment 2: Low temperature
out2 = generate(do_sample=True, temperature=0.3, top_k=50)
print("Low Temperature (0.3):")
print(out2)

Low Temperature (0.3):
Artificial Intelligence will change the future because it will be able to do things that are not possible in the past.

It's a good thing that we're not seeing that. We're seeing that there are some things that are not


In [8]:
# Experiment 3: High temperature
out3 = generate(do_sample=True, temperature=1.2, top_k=50)
print("High Temperature (1.2):")
print(out3)

High Temperature (1.2):
Artificial Intelligence will change the future because more and more people are using algorithms to solve problems they are already stuck working hard on."

So far, there is little doubt that artificial intelligence will drive some huge changes to what is seen as


In [9]:
# Experiment 4: Top-p sampling
out4 = generate(do_sample=True, top_p=0.9, top_k=0)
print("Top-p Sampling (0.9):")
print(out4)

Top-p Sampling (0.9):
Artificial Intelligence will change the future because more and more work is required in AI."


## 7. Experimental Observations

### Observation Table 1: Tokenization Comparison

In [10]:
import pandas as pd

pd.DataFrame({
    "Sr. No.": [1, 2],
    "Input Text": [text, text],
    "Tokenizer": ["BPE (tiktoken)", "WordPiece (BERT)"],
    "Number of Tokens": [len(bpe_tokens), len(wp_tokens)],
    "Tokens": [bpe_tokens, wp_tokens],
}).set_index("Sr. No.")

,Input Text,Tokenizer,Number of Tokens,Tokens
Sr. No.,,,,
1,Artificial Intelligence is transforming educat...,BPE (tiktoken),7,"[Art, ificial, Intelligence, is, transformi..."
2,Artificial Intelligence is transforming educat...,WordPiece (BERT),6,"[artificial, intelligence, is, transforming, e..."


### Observation Table 2: Decoding Parameters

In [11]:
outs = [out1, out2, out3, out4]
pd.DataFrame({
    "Experiment": [1, 2, 3, 4],
    "Parameter": ["Greedy decoding", "Temperature = 0.3", "Temperature = 1.2", "Top-p = 0.9"],
    "Generated words": [len(o[len(prompt):].split()) for o in outs],
    "Distinct words": [len(set(o[len(prompt):].lower().split())) for o in outs],
}).set_index("Experiment")

,Parameter,Generated words,Distinct words
Experiment,,,
1,Greedy decoding,36,20
2,Temperature = 0.3,33,23
3,Temperature = 1.2,36,32
4,Top-p = 0.9,8,7


**Observations**

- BPE (`tiktoken`) produced 7 tokens: `Art`, `ificial`, ` Intelligence`, ` is`, ` transforming`, ` education`, `.`. It is case sensitive, keeps the leading space as part of a token, and split the word "Artificial" into two pieces.
- WordPiece (BERT) produced 6 tokens: `artificial`, `intelligence`, `is`, `transforming`, `education`, `.`. The uncased vocabulary lower-cased the text, and every word was found as a single token.
- Greedy decoding gave a fluent but repetitive continuation ("able to do things that are not possible ... able to do things that are not possible").
- Temperature 0.3 stayed close to the greedy text. Temperature 1.2 gave the most varied text (32 distinct words out of 36) and drifted away from the opening idea.
- Top-p 0.9 produced a short output: the model sampled the end-of-text token after about 8 words.
- The effects are tendencies, not guarantees; sampled outputs change with the random seed.

## 8. Result

The tokenization strategies BPE and WordPiece were explored using Python libraries. The sentence "Artificial Intelligence is transforming education." was split into 7 BPE tokens and 6 WordPiece tokens. A pre-trained GPT-2 model was loaded successfully, and text was generated using greedy decoding, low temperature (0.3), high temperature (1.2) and top-p (0.9) sampling. The experiment demonstrated how tokenization methods and decoding settings influence text representation and generated output.

## 9. Conclusion

Tokenization converts natural language into token IDs that can be processed by language models. BPE and WordPiece use different vocabularies and splitting strategies. GPT-2 generates text autoregressively, and parameters such as temperature, top-k, top-p, and maximum new tokens affect the diversity and length of the generated text.